# Create GRPO Reasoning Mixed 3600 Dataset
**Reverse-engineered dataset builder for `abhinav0231/reasoning-mixed-3600`**

This notebook constructs the 3,600-sample prompt/ground-truth dataset used for GRPO RL training across 7 diverse reasoning domains:
1. **MATH-Hard** (800 samples) — `answer_type: exact`
2. **OpenR1-Math** (800 samples) — `answer_type: exact`
3. **GSM8K** (400 samples) — `answer_type: numeric`
4. **ARC-Challenge** (500 samples) — `answer_type: exact`
5. **StrategyQA** (400 samples) — `answer_type: bool`
6. **Alpaca** (300 samples) — `answer_type: exact`
7. **Code (HumanEval + MBPP)** (400 samples) — `answer_type: code`

The output matches the exact 5-feature Hugging Face schema:
- `prompt`: string
- `answer`: string
- `source`: string
- `answer_type`: string
- `messages`: list of `{"role": string, "content": string}`

## Cell 1 — Install Dependencies & Authentication

In [ ]:
# ==============================================================================
# Robust Authentication (Hugging Face)
# ==============================================================================
import os
try:
    from huggingface_hub import login, get_token
except ImportError:
    from huggingface_hub import login, HfFolder
    get_token = HfFolder.get_token

# Retrieve HF Token from all possible sources (os.environ, Colab Secrets, or cached token)
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN", "")
if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN") or ""
    except Exception:
        pass

if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    cached_token = get_token()
    if cached_token:
        HF_TOKEN = cached_token

if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("✅ Authenticated with Hugging Face")
    except Exception as e:
        print(f"⚠️ Hugging Face authentication note: {e}")
else:
    print("ℹ️ HF_TOKEN not provided. Proceeding (public datasets/models remain accessible).")


## Cell 2 — Configuration & Constants

In [ ]:
# ==============================================================================
# Cell 2 — Global Parameters & Domain Sample Targets
# ==============================================================================
HF_USERNAME = "abhinav0231"
OUTPUT_DATASET_REPO = f"{HF_USERNAME}/reasoning-mixed-3600"
SEED = 42

# Standardized System Prompt to instruct step-by-step reasoning & explicit answer tagging
SYSTEM_PROMPT = (
    "You are a precise, helpful assistant. "
    "Always reason step by step inside <think></think> tags, "
    "then write your final answer inside <answer></answer> tags."
)

# Itemized Target Sample Counts Across 7 Reasoning Domains (Total = 3,600 prompts)
N_MATH_HARD    = 800  # High-difficulty competition mathematics
N_OPENR1_MATH  = 800  # Open-R1 mathematical reasoning prompts
N_GSM8K        = 400  # Grade school math word problems
N_ARC          = 500  # AI2 Reasoning Challenge (science multiple-choice)
N_STRATEGY_QA  = 400  # Multi-step implicit strategy reasoning (Yes/No boolean)
N_ALPACA       = 300  # General instruction following prompts
N_CODE         = 400  # Python programming tasks (HumanEval 164 + MBPP 236)

TOTAL_TARGET = N_MATH_HARD + N_OPENR1_MATH + N_GSM8K + N_ARC + N_STRATEGY_QA + N_ALPACA + N_CODE
print(f"Target HF Repo : {OUTPUT_DATASET_REPO}")
print(f"Total Target    : {TOTAL_TARGET} samples")

## Cell 3 — Dataset Extractors for the 7 Reasoning Domains

In [ ]:
# ==============================================================================
# Cell 3 — Domain Extractor Functions & Schema Normalization
# ==============================================================================
import re
from datasets import load_dataset

# Helper function to construct standardized 5-feature dictionary schema
def make_item(prompt, answer, source, answer_type):
    return {
        "prompt": prompt.strip(),
        "answer": str(answer).strip(),
        "source": source,
        "answer_type": answer_type,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt.strip()}
        ]
    }

# ------------------------------------------------------------------------------
# 1. MATH-Hard Extractor (800 samples, answer_type: exact)
# ------------------------------------------------------------------------------
def get_math_hard(n=800):
    print(f"Extracting {n} MATH-Hard samples...")
    try:
        raw = load_dataset("DigitalLearningGmbH/MATH-lighteval", "default", split="train").shuffle(seed=SEED)
    except Exception:
        raw = load_dataset("cais/MATH", "all", split="train").shuffle(seed=SEED)
    items = []
    for ex in raw:
        prob = ex.get("problem", "")
        sol  = ex.get("solution", "")
        # Extract LaTeX boxed answer or fallback to last line of solution
        boxed = re.search(r"\\boxed\{(.+?)\}", sol)
        ans = boxed.group(1) if boxed else sol.splitlines()[-1]
        if prob and ans:
            items.append(make_item(prob, ans, "math_hard", "exact"))
        if len(items) >= n:
            break
    return items

# ------------------------------------------------------------------------------
# 2. OpenR1-Math Extractor (800 samples, answer_type: exact)
# ------------------------------------------------------------------------------
def get_openr1_math(n=800):
    print(f"Extracting {n} OpenR1-Math samples...")
    raw = load_dataset("open-r1/OpenThoughts-114k-math", split="train").shuffle(seed=SEED)
    items = []
    for ex in raw:
        msgs = ex.get("messages", [])
        u_msg = next((m["content"] for m in msgs if m["role"] == "user"), None)
        a_msg = next((m["content"] for m in msgs if m["role"] == "assistant"), None)
        if not u_msg or not a_msg:
            continue
        boxed = re.search(r"\\boxed\{(.+?)\}", a_msg)
        ans = boxed.group(1) if boxed else a_msg.splitlines()[-1]
        items.append(make_item(u_msg, ans, "openr1_math", "exact"))
        if len(items) >= n:
            break
    return items

# ------------------------------------------------------------------------------
# 3. GSM8K Extractor (400 samples, answer_type: numeric)
# ------------------------------------------------------------------------------
def get_gsm8k(n=400):
    print(f"Extracting {n} GSM8K samples...")
    raw = load_dataset("openai/gsm8k", "main", split="train").shuffle(seed=SEED)
    items = []
    for ex in raw:
        q = ex["question"]
        a_full = ex["answer"]
        # Parse numeric answer following '#### ' delimiter
        ans = a_full.split("#### ")[-1].strip() if "#### " in a_full else a_full
        items.append(make_item(q, ans, "gsm8k", "numeric"))
        if len(items) >= n:
            break
    return items

# ------------------------------------------------------------------------------
# 4. ARC-Challenge Extractor (500 samples, answer_type: exact)
# ------------------------------------------------------------------------------
def get_arc_challenge(n=500):
    print(f"Extracting {n} ARC-Challenge samples...")
    raw = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="train").shuffle(seed=SEED)
    items = []
    for ex in raw:
        q = ex["question"]
        choices = ex["choices"]
        labels = choices["label"]
        texts  = choices["text"]
        options_str = "\n".join([f"{l}. {t}" for l, t in zip(labels, texts)])
        prompt = f"{q}\nOptions:\n{options_str}"
        ansKey = ex["answerKey"]
        items.append(make_item(prompt, ansKey, "arc_challenge", "exact"))
        if len(items) >= n:
            break
    return items

# ------------------------------------------------------------------------------
# 5. StrategyQA Extractor (400 samples, answer_type: bool)
# ------------------------------------------------------------------------------
def get_strategy_qa(n=400):
    print(f"Extracting {n} StrategyQA samples...")
    raw = load_dataset("tasksource/strategy-qa", split="train").shuffle(seed=SEED)
    items = []
    for ex in raw:
        q = ex.get("question", ex.get("input", ""))
        ans_bool = ex.get("answer", ex.get("target", True))
        ans_str = "yes" if bool(ans_bool) else "no"
        if q:
            items.append(make_item(q, ans_str, "strategy_qa", "bool"))
        if len(items) >= n:
            break
    return items

# ------------------------------------------------------------------------------
# 6. Alpaca Extractor (300 samples, answer_type: exact)
# ------------------------------------------------------------------------------
def get_alpaca(n=300):
    print(f"Extracting {n} Alpaca samples...")
    raw = load_dataset("yahma/alpaca-cleaned", split="train").shuffle(seed=SEED)
    items = []
    for ex in raw:
        inst = ex["instruction"]
        inp  = ex.get("input", "")
        out  = ex["output"]
        prompt = f"{inst}\nInput: {inp}" if inp.strip() else inst
        items.append(make_item(prompt, out, "alpaca", "exact"))
        if len(items) >= n:
            break
    return items

# ------------------------------------------------------------------------------
# 7. Code Extractor (400 samples: HumanEval 164 + MBPP 236, answer_type: code)
# ------------------------------------------------------------------------------
def get_code(n=400):
    print(f"Extracting {n} Code samples (HumanEval + MBPP)...")
    items = []
    # Extract HumanEval (164 coding tasks)
    try:
        he_raw = load_dataset("openai/openai_humaneval", split="test")
        for ex in he_raw:
            items.append(make_item(ex["prompt"], ex["canonical_solution"], "humaneval", "code"))
    except Exception as e:
        print(f"  Notice on HumanEval load: {e}")
    
    # Extract MBPP to complete target count of 400
    try:
        mbpp_raw = None
        for mbpp_path in ["google-research-datasets/mbpp", "mbpp"]:
            try:
                mbpp_raw = load_dataset(mbpp_path, "sanitized", split="test").shuffle(seed=SEED)
                break
            except Exception:
                pass
        if mbpp_raw:
            for ex in mbpp_raw:
                prompt = ex.get("prompt", ex.get("text", ""))
                code = ex.get("code", "")
                if prompt and code:
                    items.append(make_item(prompt, code, "mbpp", "code"))
                if len(items) >= n:
                    break
    except Exception as e:
        print(f"  Notice on MBPP load: {e}")
        
    return items[:n]

print("✅ Extractor functions successfully defined")


## Cell 4 — Build, Merge, and Shuffle the 3,600 Dataset

In [ ]:
# ==============================================================================
# Cell 4 — Ingestion Execution, Domain Merging & Global Shuffling
# ==============================================================================
import random
from datasets import Dataset
from collections import Counter

# 1. Collect samples across all 7 reasoning domains
math_hard_samples  = get_math_hard(N_MATH_HARD)
openr1_samples     = get_openr1_math(N_OPENR1_MATH)
gsm8k_samples      = get_gsm8k(N_GSM8K)
arc_samples        = get_arc_challenge(N_ARC)
strategyqa_samples = get_strategy_qa(N_STRATEGY_QA)
alpaca_samples     = get_alpaca(N_ALPACA)
code_samples       = get_code(N_CODE)

# 2. Concatenate into single list
all_data = (
    math_hard_samples +
    openr1_samples +
    gsm8k_samples +
    arc_samples +
    strategyqa_samples +
    alpaca_samples +
    code_samples
)

# 3. Shuffle globally with fixed seed
random.seed(SEED)
random.shuffle(all_data)

# 4. Construct Hugging Face Dataset
dataset = Dataset.from_list(all_data)

print(f"\n✅ Successfully constructed {len(dataset):,} samples dataset!")
print(f"Columns: {dataset.column_names}")

# 5. Display statistical distribution across domain sources and answer types
source_counts = Counter(dataset["source"])
type_counts   = Counter(dataset["answer_type"])

print(f"\n--- Source Distribution ---")
for k, v in source_counts.items():
    print(f"  {k:<15}: {v:>4}")

print(f"\n--- Answer Type Distribution ---")
for k, v in type_counts.items():
    print(f"  {k:<15}: {v:>4}")

## Cell 5 — Verify Dataset Sample Schema

In [ ]:
# ==============================================================================
# Cell 5 — Schema Integrity & Column Verification
# ==============================================================================
print("Sample index 0 preview:")
sample0 = dataset[0]
print(f"Prompt      : {sample0['prompt'][:120]}...")
print(f"Answer      : {sample0['answer'][:120]}...")
print(f"Source      : {sample0['source']}")
print(f"Answer Type : {sample0['answer_type']}")
print(f"Messages    : {sample0['messages']}")

# Assert structural integrity
assert len(dataset) == 3600, f"Expected 3600 samples, got {len(dataset)}"
assert set(dataset.column_names) == {"prompt", "answer", "source", "answer_type", "messages"}, "Schema mismatch!"
print("\n✅ Schema and sample count verified successfully!")

## Cell 6 — Push to Hugging Face Dataset Hub

In [ ]:
# ==============================================================================
# Cell 6 — Upload Preprocessed GRPO Dataset to Hugging Face Hub
# ==============================================================================
print(f"Pushing reasoning-mixed-3600 dataset to Hugging Face: {OUTPUT_DATASET_REPO} ...")
dataset.push_to_hub(OUTPUT_DATASET_REPO, private=False)
print(f"\n🎉 Successfully pushed dataset to:")
print(f"   https://huggingface.co/datasets/{OUTPUT_DATASET_REPO}")